# Medallion model consistency audit

Read-only: this notebook does not create, alter or delete tables or constraints. Run after Bronze, Silver and Gold are complete. It checks inventory, grain, preservation across layers, relationships and exceptions, product hierarchy and EDA reconciliation. Relationships used by code are distinguished from constraints registered in Catalog. Local fixture checks do not replace this remote audit.

In [ ]:
from pipeline_support import load_config, PipelineContext
CFG=load_config()
lineage=PipelineContext(spark,CFG,'gold')
validated_versions=None
if spark.catalog.tableExists(lineage.state_table):
    validated_versions=lineage.require_complete('gold')
else:
    print('Modo legado: no hay registro de ejecuciones; esta auditoría no acredita el cierre de la cadena.')
from pyspark.sql import functions as F
from decimal import Decimal
K=['CheckId','LocationId','BusinessDate']
B='workspace.applebees_bronze';S='workspace.applebees_silver';G='workspace.applebees_gold'
KEYS={
 'dim_locations':['Id'],'dim_items':['Id'],'dim_revenue_centers':['Id'],'dim_employees':['Id'],
 'checks':K,'items_sold':K+['item_pos'],'payments':K+['payment_pos'],'taxes':K+['tax_pos'],
 'voids':K+['void_pos'],'item_quality':K+['item_pos'],'ticket_quality':K,'reference_quality':['check_name']}
GKEYS={'ticket_metrics':K,'daily_location_channel':['BusinessDate','LocationId','RevenueCenterId'],
 'employee_ticket_activity':K+['EmployeeId'],
 'employee_activity_daily':['BusinessDate','LocationId','RevenueCenterId','EmployeeId'],
 'review_thresholds':['RevenueCenterId'],'review_cases':K,
 'discount_type_daily':['BusinessDate','LocationId','RevenueCenterId','discount_scope','DiscountId','DiscountName']}
TABLES=[B+'.'+x+'_raw' for x in ['checks','locations','items','revenue_centers','employees']]+[S+'.'+x for x in list(KEYS)+['discounts']]+[G+'.'+x for x in GKEYS]
missing=[t for t in TABLES if not spark.catalog.tableExists(t)]
assert not missing,f'Tablas pendientes: {missing}'
print('25 tablas esperadas presentes. Ninguna tabla será modificada.')
audit=[]
def record(name,status,details):audit.append((name,status,str(details)))


## 1. Keys and grain

Check data uniqueness even without declared primary keys. Validate discounts by scope because item_pos is null for ticket discounts. A unique aggregate key can contain NULL grouping values, preventing a PK declaration without proving duplication.

In [ ]:
for schema,spec in [(S,KEYS),(G,GKEYS)]:
    for table,key in spec.items():
        df=spark.table(schema+'.'+table)
        n_null=df.filter(' OR '.join(f'`{c}` IS NULL' for c in key)).count()
        n_duplicates=df.groupBy(*key).count().filter('count>1').count()
        status='OK' if not n_null and not n_duplicates else ('PENDIENTE_PK_NULOS' if schema==G and table=='discount_type_daily' and not n_duplicates else 'ERROR')
        record(schema+'.'+table,status,f'Llave={key}; filas con nulos={n_null}; grupos repetidos={n_duplicates}')
for scope,key in [('ticket',K+['discount_pos']),('item',K+['item_pos','discount_pos'])]:
    df=spark.table(S+'.discounts').filter(F.col('discount_scope')==scope)
    bad=df.filter(' OR '.join(f'`{c}` IS NULL' for c in key)).count()
    dup=df.groupBy(*key).count().filter('count>1').count()
    record('discounts '+scope,'OK' if bad==0 and dup==0 else 'ERROR',f'{bad} llaves nulas; {dup} grupos duplicados')


## 2. Bronze → Silver → Gold preservation

Equal row counts are insufficient: compare ticket keys, child positions, array sizes and summary amounts. Silver dimensions preserve validated inputs rather than performing additional cleanup.

In [ ]:
raw=spark.table(B+'.checks_raw')
checks=spark.table(S+'.checks')
bronze_keys=raw.select(F.col('Id').alias('CheckId'),'LocationId',F.col('BusinessDate').cast('date').alias('BusinessDate'))
for name,df in [('Silver checks',checks),('Gold ticket_metrics',spark.table(G+'.ticket_metrics')),('Silver ticket_quality',spark.table(S+'.ticket_quality'))]:
    observed=df.select(*K)
    lost=bronze_keys.exceptAll(observed).count();extra=observed.exceptAll(bronze_keys).count()
    record('Llaves Bronze vs '+name,'OK' if lost==extra==0 else 'ERROR',f'Faltan={lost}, sobran={extra}')
coverage=checks.agg(F.count('*').alias('tickets'),F.countDistinct('BusinessDate').alias('days'),F.min('BusinessDate').alias('first'),F.max('BusinessDate').alias('last')).first()
record('Cobertura vigente','OK' if coverage.tickets>0 and coverage.days>0 else 'ERROR',coverage)
if CFG['run_original_regression']:
    original=CFG['original']
    record('Cobertura original','OK' if coverage.tickets==original['tickets'] and coverage.days==original['days'] and str(coverage.first)==original['first_date'] and str(coverage.last)==original['last_date'] else 'ERROR',coverage)
for field,table,pos in [('ItemsSold','items_sold','item_pos'),('Payments','payments','payment_pos'),('Discounts',None,'discount_pos'),('Tax','taxes','tax_pos'),('Voids','voids','void_pos')]:
    expected=raw.select(F.when(F.col(field).isNull(),0).otherwise(F.size(field)).alias('n')).agg(F.sum('n')).first()[0]
    df=spark.table(S+'.'+table) if table else spark.table(S+'.discounts').filter("discount_scope='ticket'")
    record('Filas array '+field,'OK' if df.count()==expected else 'ERROR',f'Esperadas={expected}, observadas={df.count()}')
    expected_keys=raw.select(F.col('Id').alias('CheckId'),'LocationId',F.col('BusinessDate').cast('date').alias('BusinessDate'),F.posexplode(field).alias(pos,'record')).select(*(K+[pos]))
    actual_keys=df.select(*(K+[pos]))
    lost=expected_keys.exceptAll(actual_keys).count();extra=actual_keys.exceptAll(expected_keys).count()
    record('Posiciones '+field,'OK' if lost==extra==0 else 'ERROR',f'Faltan={lost}, sobran={extra}')
nested=raw.select(F.col('Id').alias('CheckId'),'LocationId',F.col('BusinessDate').cast('date').alias('BusinessDate'),F.posexplode('ItemsSold').alias('item_pos','item'))
line_keys=nested.select(*K,'item_pos',F.posexplode('item.Discounts').alias('discount_pos','record')).select(*(K+['item_pos','discount_pos']))
observed=spark.table(S+'.discounts').filter("discount_scope='item'").select(*(K+['item_pos','discount_pos']))
lost=line_keys.exceptAll(observed).count();extra=observed.exceptAll(line_keys).count()
record('Descuentos de producto conservados','OK' if lost==extra==0 else 'ERROR',f'Faltan={lost}, sobran={extra}')
for name in ['locations','items','revenue_centers','employees']:
    a=spark.table(B+'.'+name+'_raw');b=spark.table(S+'.dim_'+name)
    same_columns=set(a.columns)==set(b.columns)
    diff=(a.exceptAll(b.select(*a.columns)).limit(1).count()+b.select(*a.columns).exceptAll(a).limit(1).count()) if same_columns else -1
    record('Dimensión '+name,'OK' if same_columns and diff==0 else 'ERROR','Copia íntegra de entrada' if diff==0 else 'Contenido/columnas difieren')


## 3. Validate relationships before declaring foreign keys

Employee, manager and home-location references may contain special values such as 0. Report unmatched counts and examples without deleting or changing them. Structural ticket/line relationships are mandatory.

In [ ]:
def relation(child,child_cols,parent,parent_cols,label,required=False,condition=None):
    left=spark.table(child)
    if condition:left=left.filter(condition)
    right=spark.table(parent)
    a={f.name:f.dataType for f in left.schema.fields};b={f.name:f.dataType for f in right.schema.fields}
    types_ok=all(a[c]==b[p] for c,p in zip(child_cols,parent_cols))
    nulls=left.filter(' OR '.join(f'`{c}` IS NULL' for c in child_cols)).count()
    observed=left.select(*child_cols).filter(' AND '.join(f'`{c}` IS NOT NULL' for c in child_cols))
    parents=right.select(*[F.col(p).alias(c) for c,p in zip(child_cols,parent_cols)])
    missing=observed.join(parents,child_cols,'left_anti')
    count=missing.count()
    status='ERROR' if not types_ok or (required and (count or nulls)) else ('PENDIENTE_REFERENCIAS' if count else 'OK')
    record(label,status,f'Tipos compatibles={types_ok}; nulas={nulls}; sin padre={count}; muestra={[x.asDict() for x in missing.limit(3).collect()]}')
for name in ['items_sold','payments','discounts','taxes','voids','ticket_quality']:
    relation(S+'.'+name,K,S+'.checks',K,name+' -> ticket',True)
relation(S+'.item_quality',K+['item_pos'],S+'.items_sold',K+['item_pos'],'item_quality -> línea',True)
relation(S+'.discounts',K+['item_pos'],S+'.items_sold',K+['item_pos'],'descuento producto -> línea',True,"discount_scope='item'")
for child,col,parent in [('checks','LocationId','dim_locations'),('checks','RevenueCenterId','dim_revenue_centers'),('items_sold','ItemId','dim_items'),('items_sold','RevenueCenterId','dim_revenue_centers'),('voids','ItemId','dim_items'),('dim_employees','LocationId','dim_locations'),('dim_employees','HomeLocationId','dim_locations')]:
    relation(S+'.'+child,[col],S+'.'+parent,['Id'],child+'.'+col+' -> '+parent)
for name in ['items_sold','payments','discounts','voids']:
    relation(S+'.'+name,['EmployeeId'],S+'.dim_employees',['Id'],name+' -> empleado')
for name in ['discounts','voids']:
    relation(S+'.'+name,['ManagerId'],S+'.dim_employees',['Id'],name+' -> gerente')
relation(G+'.ticket_metrics',K,S+'.checks',K,'Gold ticket_metrics -> Silver checks',True)
for name in ['employee_ticket_activity','review_cases']:
    relation(G+'.'+name,K,G+'.ticket_metrics',K,name+' -> Gold ticket',True)
for name in ['ticket_metrics','daily_location_channel','employee_ticket_activity','employee_activity_daily','discount_type_daily','review_cases']:
    relation(G+'.'+name,['LocationId'],S+'.dim_locations',['Id'],name+' -> sucursal')
for name in ['ticket_metrics','daily_location_channel','employee_ticket_activity','employee_activity_daily','discount_type_daily','review_thresholds','review_cases']:
    relation(G+'.'+name,['RevenueCenterId'],S+'.dim_revenue_centers',['Id'],name+' -> canal')
for name in ['employee_ticket_activity','employee_activity_daily']:
    relation(G+'.'+name,['EmployeeId'],S+'.dim_employees',['Id'],name+' -> empleado')
for name in ['ticket_metrics','review_cases']:
    relation(G+'.'+name,['payment_employee_id'],S+'.dim_employees',['Id'],name+' -> empleado de pago')


## 4. Product hierarchy

Sequence and ParentSequence describe POS relationships. First check Sequence uniqueness within ticket and POS source. ParentSequence=0 is treated as a root for exploration. Confirm business semantics before declaring an FK, even if counts show no errors.

In [ ]:
items=spark.table(S+'.items_sold')
sequence_key=K+['PosSourceId','Sequence']
duplicates=items.groupBy(*sequence_key).count().filter('count>1').count()
parents=items.select(*K,'PosSourceId',F.col('Sequence').alias('ParentSequence')).distinct()
child=items.filter('ParentSequence>0').select(*K,'PosSourceId','ParentSequence')
missing=child.join(parents,K+['PosSourceId','ParentSequence'],'left_anti').count()
record('Jerarquía POS','PENDIENTE_DEFINICION',f'Grupos Sequence duplicados={duplicates}; referencias ParentSequence positivas sin padre={missing}. No se declara FK automáticamente.')
display(items.groupBy(*sequence_key).count().filter('count>1').limit(10))


## 5. Gold versus detail and executed EDA

Historical expected totals refer to the reviewed complete export. Differences stop confirmation when original regression is enabled. They are not tests of transaction legitimacy.

In [ ]:
daily=spark.table(G+'.daily_location_channel')
row=daily.agg(F.sum('tickets').alias('tickets'),F.sum('discounted_tickets').alias('discounted'),F.sum('tickets_with_voids').alias('voids'),F.sum('non_reconciled_tickets').alias('non_reconciled'),F.sum('mismatched_item_lines').alias('mismatches'),F.sum('net_sales_reconciled').alias('sales'),F.sum('discounts_reconciled').alias('discounts')).first()
if CFG['run_original_regression']:
    for name,expected in [('tickets',2142846),('discounted',268580),('voids',55120),('non_reconciled',59),('mismatches',89)]:
        record('Gold vs EDA '+name,'OK' if row[name]==expected else 'ERROR',f'{row[name]} vs {expected}')
    for name,expected in [('sales',Decimal('84292048.6000')),('discounts',Decimal('2968583.3900'))]:
        record('Gold vs EDA '+name,'OK' if row[name] is not None and abs(row[name]-expected)<=Decimal(CFG['tolerance']) else 'ERROR',f'{row[name]} vs {expected}')
else:
    record('Gold vs EDA original','NO_APLICA','Regresión histórica desactivada para cargas nuevas/corregidas')
source=spark.table(S+'.discounts').count();aggregated=spark.table(G+'.discount_type_daily').agg(F.sum('applications')).first()[0] or 0
record('Aplicaciones descuentos detalle vs Gold','OK' if source==aggregated else 'ERROR',f'{source} vs {aggregated}')
source=spark.table(G+'.employee_ticket_activity').count();aggregated=spark.table(G+'.employee_activity_daily').agg(F.sum('associated_tickets')).first()[0] or 0
record('Actividad empleado detalle vs Gold','OK' if source==aggregated else 'ERROR',f'{source} vs {aggregated}')


## 6. Declared constraints and missing reference catalogs

Report constraints actually registered in Catalog. An Id suffix does not guarantee an available dimension. No independent discount, tax, void-category or job-position dimensions were supplied; do not invent parent tables.

In [ ]:
display(spark.sql("""SELECT table_schema,table_name,constraint_name,constraint_type
FROM workspace.information_schema.table_constraints
WHERE table_schema IN ('applebees_silver','applebees_gold') ORDER BY table_schema,table_name,constraint_type"""))
for table in ['checks','items_sold','payments','discounts','taxes','voids','dim_employees','dim_items']:
    columns=[c for c in spark.table(S+'.'+table).columns if c.lower().endswith('id')]
    print(table,columns)
print('DiscountId, TaxCategoryId, VoidCategoryId, PaymentCategoryId, categorías de producto, puestos y LoyaltyId: documentar códigos, no declarar FK sin catálogo padre validado.')
result=spark.createDataFrame(audit,'check_name STRING,status STRING,details STRING')
display(result)
display(result.groupBy('status').count())
if validated_versions is not None:
    assert lineage.require_complete('gold') == validated_versions, 'Las versiones cambiaron durante la auditoría; repetirla.'
failures=[x for x in audit if x[1]=='ERROR']
assert not failures,f'Hay {len(failures)} errores estructurales/contables. No declarar restricciones ni aprobar la cadena hasta revisarlos.'
print('Sin errores estructurales detectados. Revisar estados PENDIENTE antes de afirmar que todas las relaciones están cerradas.')


## Scope

Review audit outputs and exceptions. Relationships are handled by the validated Silver/Gold blocks; this package does not require an additional standalone relationship notebook. The 59 nonreconciled tickets, 89 discrepant lines and special employee/location values were not automatically corrected. They remain evidence; the model does not establish operational correctness for every record.